# **<span style="color:#48B3AF; font-family: 'Courier New"> TRANSFORM </span>**

## **<span style="color:#476EAE; font-family: 'Courier New"> Resources </span>**

In [160]:
# Infrastructure
import os
import pandas as pd
import numpy as np

# Locations
os.makedirs("loading", exist_ok=True)

# Mappings
bool_map = {"SI": True, "Si": True, "NO": False, "No": False}
compensation_map = {"1": "Full", "integral": "Full"}

### **<span style="color:#A7E399; font-family: 'Courier New"> Republic of Moldova </span>**

In [161]:
# Load files for transformation
lista_dc_cnam = pd.read_csv("transforming/lista_dc_cnam.csv", dtype=str)
dci = pd.read_csv("transforming/dci.csv", dtype=str)
cim = pd.read_csv("transforming/cim.csv", dtype=str)

In [162]:
# Remove leading and trailing spaces
lista_dc_cnam = lista_dc_cnam.map(lambda x: x.strip() if isinstance(x, str) else x)
dci = dci.map(lambda x: x.strip() if isinstance(x, str) else x)
cim = cim.map(lambda x: x.strip() if isinstance(x, str) else x)

In [163]:
display(lista_dc_cnam.head(2))

,Grupa maladiilor pentru compensare,Cod DCI,Denumirea Comuna Internationala (DCI),Doza în SI MC,Cod DC,Denumirea comercială (DC),Doza,Forma farmaceutică,Divizarea,Ţara,Firma producătoare,Număr de înregistrare,Data înregistrării,Cod ATC,Cod medicament (Catalogul național de prețuri),is_child_rate,coverage_type,Cod DCI_sec,Cod DC_sec
0,Pentru tratamentul maladiilor aparatului locom...,060,ACECLOFENACUM,100,0601001,Aceclofenac-BP,100 mg,comprimate,N10x3,Republica Moldova,"Balkan Pharmaceuticals SRL, SC",26519,10.09.2020,M01AB16,9200900714,False,Partial,NaN,NaN
1,Pentru tratamentul maladiilor aparatului locom...,060,ACECLOFENACUM,100,0601002,Airtal,100 mg,comp. film.,N10x6,Ungaria,Gedeon Richter PLC,25086,29.10.2018,M01AB16,0109370013,False,Partial,NaN,NaN


In [164]:
display(dci.head(2))

,Nr. d/o,Denumirea comună internațională (DCI),Doza,Forma farmaceutică,Rata de compensare din mediana prețurilor,"Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA","Suma fixă compensată per unitate de măsură (MDL), fără TVA",Cod CIM \nde diagnostic/limitare indicație,Durata tratamentului,Destinat
0,1.0,ACECLOFENACUM,100 mg,"comprimate, comprimate filmate",1,2.33,2.16,"M069-Artrita reumatoida, fara precizare\nM199-...",cronic,adulți/copii
1,NaN,ACECLOFENACUM,100 mg,"comprimate, comprimate filmate",1,2.33,2.16,"M069-Artrita reumatoida, fara precizare\nM109-...",episodic,adulți


In [17]:
display(cim.head(2))

,CodDiagnostic,DenDiagnostic,DenSubClasaDiagnostic,DenClasaDiagnostic
0,A000,"Holera cu Vibrio cholerae,biovar cholerae",Boli infectioase intestinale,Boli infectioase si parazitare
1,A001,"Holera cu Vibrio cholerae eltor,biovar eltor",Boli infectioase intestinale,Boli infectioase si parazitare


In [165]:
dc_dci_coverage = pd.merge(lista_dc_cnam, dci[["Denumirea comună internațională (DCI)", "Doza", "Forma farmaceutică", "Rata de compensare din mediana prețurilor", "Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA", "Suma fixă compensată per unitate de măsură (MDL), fără TVA", "Cod CIM \nde diagnostic/limitare indicație", "Durata tratamentului", "Destinat"]], left_on='Denumirea Comuna Internationala (DCI)', right_on='Denumirea comună internațională (DCI)', how='outer', suffixes=("_dc", "_dci"))

In [166]:
dc_dci_coverage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1479 entries, 0 to 1478
Data columns (total 28 columns):
 #   Column                                                          Non-Null Count  Dtype 
---  ------                                                          --------------  ----- 
 0   Grupa maladiilor pentru compensare                              1467 non-null   object
 1   Cod DCI                                                         1265 non-null   object
 2   Denumirea Comuna Internationala (DCI)                           1467 non-null   object
 3   Doza în SI MC                                                   1467 non-null   object
 4   Cod DC                                                          1266 non-null   object
 5   Denumirea comercială (DC)                                       1467 non-null   object
 6   Doza_dc                                                         1468 non-null   object
 7   Forma farmaceutică_dc                                       

In [167]:
dc_dci_coverage.head(2)

,Grupa maladiilor pentru compensare,Cod DCI,Denumirea Comuna Internationala (DCI),Doza în SI MC,Cod DC,Denumirea comercială (DC),Doza_dc,Forma farmaceutică_dc,Divizarea,Ţara,...,Cod DC_sec,Denumirea comună internațională (DCI),Doza_dci,Forma farmaceutică_dci,Rata de compensare din mediana prețurilor,"Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA","Suma fixă compensată per unitate de măsură (MDL), fără TVA",Cod CIM \nde diagnostic/limitare indicație,Durata tratamentului,Destinat
0,Pentru tratamentul maladiilor aparatului locom...,060,ACECLOFENACUM,100,0601001,Aceclofenac-BP,100 mg,comprimate,N10x3,Republica Moldova,...,NaN,ACECLOFENACUM,100 mg,"comprimate, comprimate filmate",1,2.33,2.16,"M069-Artrita reumatoida, fara precizare\nM199-...",cronic,adulți/copii
1,Pentru tratamentul maladiilor aparatului locom...,060,ACECLOFENACUM,100,0601001,Aceclofenac-BP,100 mg,comprimate,N10x3,Republica Moldova,...,NaN,ACECLOFENACUM,100 mg,"comprimate, comprimate filmate",1,2.33,2.16,"M069-Artrita reumatoida, fara precizare\nM109-...",episodic,adulți


In [195]:
# Select and rearrange necessary fields
dc_dci_coverage = dc_dci_coverage.reindex(columns=[
    'Cod DCI',
    'Denumirea Comuna Internationala (DCI)',
    'Cod DCI_sec',
    'Denumirea comună internațională (DCI)',
    'Cod DC',
    'Cod DC_sec',
    'Denumirea comercială (DC)',
    'Cod ATC',
    'Cod medicament (Catalogul național de prețuri)',
    'Număr de înregistrare',
    'Forma farmaceutică_dc',
    'Forma farmaceutică_dci',
    'coverage_type',
    'Rata de compensare din mediana prețurilor',
    'is_child_rate',
    'Grupa maladiilor pentru compensare',
    'Cod CIM \nde diagnostic/limitare indicație',
    'Durata tratamentului',
    'Doza în SI MC',
    'Doza_dci',
    'Doza_dc', 
    'Divizarea',
    'Firma producătoare', 
    'Ţara', 
    'Data înregistrării', 
    'Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA',
    'Suma fixă compensată per unitate de măsură (MDL), fără TVA', 
    'Destinat'
])


In [196]:
# Set numeric values
dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA'] = dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA'].replace('nan', np.nan).replace('-', np.nan)
dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA'] = dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA'].fillna(0).astype(float)

dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), fără TVA'] = dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), fără TVA'].replace('nan', np.nan).replace('-', np.nan)
dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), fără TVA'] = dc_dci_coverage['Suma fixă compensată per unitate de măsură (MDL), fără TVA'].fillna(0).astype(float)

In [197]:
display(dc_dci_coverage.dropna(how='all'))

,Cod DCI,Denumirea Comuna Internationala (DCI),Cod DCI_sec,Denumirea comună internațională (DCI),Cod DC,Cod DC_sec,Denumirea comercială (DC),Cod ATC,Cod medicament (Catalogul național de prețuri),Număr de înregistrare,...,Doza în SI MC,Doza_dci,Doza_dc,Divizarea,Firma producătoare,Ţara,Data înregistrării,"Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA","Suma fixă compensată per unitate de măsură (MDL), fără TVA",Destinat
0,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601001,NaN,Aceclofenac-BP,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x3,"Balkan Pharmaceuticals SRL, SC",Republica Moldova,10.09.2020,2.33,2.16,adulți/copii
1,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601001,NaN,Aceclofenac-BP,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x3,"Balkan Pharmaceuticals SRL, SC",Republica Moldova,10.09.2020,2.33,2.16,adulți
2,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601002,NaN,Airtal,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x6,Gedeon Richter PLC,Ungaria,29.10.2018,2.33,2.16,adulți/copii
3,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601002,NaN,Airtal,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x6,Gedeon Richter PLC,Ungaria,29.10.2018,2.33,2.16,adulți
4,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601004,NaN,Infenac,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10,"Tulip Lab Private Limited, India(PROD: Tulip L...",India,10.09.2020,2.33,2.16,adulți/copii
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,NaN
1475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,,NaN,NaN,NaN,NaN,0.00,0.00,NaN
1476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,NaN
1477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,NaN


In [198]:
dc_dci_coverage.head()

,Cod DCI,Denumirea Comuna Internationala (DCI),Cod DCI_sec,Denumirea comună internațională (DCI),Cod DC,Cod DC_sec,Denumirea comercială (DC),Cod ATC,Cod medicament (Catalogul național de prețuri),Număr de înregistrare,...,Doza în SI MC,Doza_dci,Doza_dc,Divizarea,Firma producătoare,Ţara,Data înregistrării,"Suma fixă compensată per unitate de măsură (MDL), inclusiv TVA","Suma fixă compensată per unitate de măsură (MDL), fără TVA",Destinat
0,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601001,NaN,Aceclofenac-BP,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x3,"Balkan Pharmaceuticals SRL, SC",Republica Moldova,10.09.2020,2.33,2.16,adulți/copii
1,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601001,NaN,Aceclofenac-BP,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x3,"Balkan Pharmaceuticals SRL, SC",Republica Moldova,10.09.2020,2.33,2.16,adulți
2,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601002,NaN,Airtal,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x6,Gedeon Richter PLC,Ungaria,29.10.2018,2.33,2.16,adulți/copii
3,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601002,NaN,Airtal,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10x6,Gedeon Richter PLC,Ungaria,29.10.2018,2.33,2.16,adulți
4,060,ACECLOFENACUM,NaN,ACECLOFENACUM,0601004,NaN,Infenac,M01AB16,NaN,NaN,...,100,100 mg,100 mg,N10,"Tulip Lab Private Limited, India(PROD: Tulip L...",India,10.09.2020,2.33,2.16,adulți/copii


In [200]:
# Selecting necessary fields
dc_dci_coverage = dc_dci_coverage[[
    'Cod DCI', 
    'Denumirea Comuna Internationala (DCI)', 
    'Cod DCI_sec',
    'Denumirea comună internațională (DCI)',
    'Cod DC', 
    'Cod DC_sec',
    'Denumirea comercială (DC)', 
    'Cod ATC', 
    'coverage_type',
    'Grupa maladiilor pentru compensare',
    'Cod CIM \nde diagnostic/limitare indicație',
    'Firma producătoare', 
    'Ţara'
]]

In [201]:
# Save file for LOAD step
dc_dci_coverage.to_csv("loading/dc_dci_coverage.csv", header = True, index = False)

### **<span style="color:#A7E399; font-family: 'Courier New"> Spain </span>**

In [56]:
# Load files for transformation
df_presentation = pd.read_csv("transforming/df_presentation.csv", dtype=str)
df_formulation = pd.read_csv("transforming/df_formulation.csv", dtype=str)
df_coverage = pd.read_csv("transforming/df_coverage.csv", dtype=str)

In [57]:
# Remove leading and trailing spaces
df_presentation = df_presentation.map(lambda x: x.strip() if isinstance(x, str) else x)
df_formulation = df_formulation.map(lambda x: x.strip() if isinstance(x, str) else x)
df_coverage = df_coverage.map(lambda x: x.strip() if isinstance(x, str) else x)

In [ ]:
# Boolean transformations
for col in [
    "¿Comercializado?",
    "¿Triangulo Amarillo?",
    "¿Problemas de suministro?",
    "¿Sustituible?",
    "¿Afecta conducción?"
]:
    if col in df_presentation.columns:
        df_presentation[col] = df_presentation[col].map(bool_map)

In [59]:
display(df_presentation.head(2))

,Nº Registro,Cod. Nacional,Presentación,Laboratorio,Fecha Aut.,Estado,Fecha Estado,Cód. ATC,Principios Activos,¿Comercializado?,¿Triangulo Amarillo?,Observaciones,¿Sustituible?,¿Afecta conducción?,¿Problemas de suministro?
0,81510,713926,"ATROPINA AGUETTANT 0,1 MG/ML SOLUCION INYECTAB...",Laboratoire Aguettant,09/12/2016,Autorizado,01/02/2017,A03BA01,ATROPINA SULFATO,True,False,Uso Hospitalario,NaN,True,False
1,81513,713937,"ATROPINA AGUETTANT 0,2 MG/ML SOLUCION INYECTAB...",Laboratoire Aguettant,12/12/2016,Autorizado,22/02/2017,A03BA01,ATROPINA SULFATO,True,False,Uso Hospitalario,NaN,True,False


In [60]:
for col in [
    "¿Comercializado?",
    "¿Triangulo Amarillo?",
    "¿Problemas de suministro?",
    "¿Sustituible?",
    "¿Afecta conducción?"
]:
    if col in df_formulation.columns:
        df_formulation[col] = df_formulation[col].map(bool_map)

In [62]:
display(df_formulation.head(2))

,Nº Registro,Medicamento,Laboratorio,Fecha Aut.,Estado,Fecha Estado,Cód. ATC,Principios Activos,Nº P. Activos,¿Comercializado?,¿Triangulo Amarillo?,Observaciones,¿Sustituible?,¿Afecta conducción?,¿Problemas de suministro?
0,88639,2LARTH GLÓBULOS,"Labo´Life España, S.A.",15/02/2023,Autorizado,16/02/2023,V03AX,"INTERLEUCINA 1 C10, INTERLEUCINA 1 C17, INTERL...",12,True,False,Sin Receta,NaN,False,False
1,88647,2LC1 GLÓBULOS,"Labo´Life España, S.A.",15/02/2023,Autorizado,16/02/2023,V03AX,"DESOXIRRIBONUCLEICO ACIDO K200, INTERLEUCINA 1...",41,True,False,Sin Receta,NaN,False,False


In [63]:
for col in [
    "Situación de financiación"
]:
    if col in df_coverage.columns:
        df_coverage[col] = df_coverage[col].map(bool_map)

In [64]:
display(df_coverage.head(2))

,Código nacional,Principio activo o asociación*,Nombre del medicamento,Situación de financiación,Tipo de medicamento,Más Información
0,600001,DOXAZOSINA,DOXAZOSINA NEO TEVA-RATIOPHARM 4 mg COMPRIMIDO...,True,Genérico,NaN
1,600002,ACETILCISTEINA,ACETILCISTEINA PENSA 600 mg POLVO PARA SOLUCIO...,True,Genérico,NaN


In [188]:
df_full = pd.merge(df_presentation, df_coverage, left_on='Cod. Nacional', right_on='Código nacional', how='outer')
df_full = pd.merge(df_full, df_formulation[["Nº Registro", "Medicamento", "Laboratorio", "Fecha Aut.", "Estado",  "Fecha Estado", "Cód. ATC", "Principios Activos", "Nº P. Activos"]], left_on='Nº Registro', right_on='Nº Registro', how='outer', suffixes=('_pres', '_form'))

In [181]:
df_full.columns

Index(['Nº Registro', 'Cod. Nacional', 'Presentación', 'Laboratorio_pres',
       'Fecha Aut._pres', 'Estado_pres', 'Fecha Estado_pres', 'Cód. ATC_pres',
       'Principios Activos_pres', '¿Comercializado?', '¿Triangulo Amarillo?',
       'Observaciones', '¿Sustituible?', '¿Afecta conducción?',
       '¿Problemas de suministro?', 'Código nacional',
       'Principio activo o asociación*', 'Nombre del medicamento',
       'Situación de financiación', 'Tipo de medicamento', 'Más Información',
       'Medicamento', 'Laboratorio_form', 'Fecha Aut._form', 'Estado_form',
       'Fecha Estado_form', 'Cód. ATC_form', 'Principios Activos_form',
       'Nº P. Activos'],
      dtype='object')

In [182]:
# Set numeric values
df_full['Nº P. Activos'] = df_full['Nº P. Activos'].replace('nan', np.nan)
df_full['Nº P. Activos'] = df_full['Nº P. Activos'].fillna(0).astype(int)
# Set date values
df_full['Fecha Aut._pres'] = pd.to_datetime(df_full['Fecha Aut._pres'], dayfirst=True)
df_full['Fecha Estado_pres'] = pd.to_datetime(df_full['Fecha Estado_pres'], dayfirst=True)
df_full['Fecha Aut._form'] = pd.to_datetime(df_full['Fecha Aut._form'], dayfirst=True)
df_full['Fecha Estado_form'] = pd.to_datetime(df_full['Fecha Estado_form'], dayfirst=True)

In [183]:
df_full.sort_values(by='Presentación').head(2)

,Nº Registro,Cod. Nacional,Presentación,Laboratorio_pres,Fecha Aut._pres,Estado_pres,Fecha Estado_pres,Cód. ATC_pres,Principios Activos_pres,¿Comercializado?,...,Tipo de medicamento,Más Información,Medicamento,Laboratorio_form,Fecha Aut._form,Estado_form,Fecha Estado_form,Cód. ATC_form,Principios Activos_form,Nº P. Activos
62888,88639,762106,"2LARTH glóbulos, 30 capsulas","Labo´Life España, S.A.",2023-02-15,Autorizado,2023-02-15,V03AX,"INTERLEUCINA 1 C10, INTERLEUCINA 1 C17, INTERL...",True,...,NaN,NaN,2LARTH GLÓBULOS,"Labo´Life España, S.A.",2023-02-15,Autorizado,2023-02-16,V03AX,"INTERLEUCINA 1 C10, INTERLEUCINA 1 C17, INTERL...",12
62897,88647,762116,"2LC1 glóbulos,30 capsulas","Labo´Life España, S.A.",2023-02-15,Autorizado,2023-02-15,V03AX,"DESOXIRRIBONUCLEICO ACIDO K200, INTERLEUCINA 1...",True,...,NaN,NaN,2LC1 GLÓBULOS,"Labo´Life España, S.A.",2023-02-15,Autorizado,2023-02-16,V03AX,"DESOXIRRIBONUCLEICO ACIDO K200, INTERLEUCINA 1...",41


In [184]:
# adjusting data types
df_full = df_full.astype({
    'Nº Registro': str, 
    'Cod. Nacional': str, 
    'Presentación': str, 
    'Laboratorio_pres': str,
    'Estado_pres': str, 
    'Cód. ATC_pres': str,
    'Principios Activos_pres': str, 
    '¿Comercializado?': bool, 
    '¿Triangulo Amarillo?': bool,
    'Observaciones': str, 
    '¿Sustituible?': bool, 
    '¿Afecta conducción?': bool,
    '¿Problemas de suministro?': bool, 
    'Código nacional': str,
    'Principio activo o asociación*': str, 
    'Nombre del medicamento': str,
    'Situación de financiación': str, 
    'Tipo de medicamento': str, 
    'Más Información': str,
    'Medicamento': str, 
    'Laboratorio_form': str, 
    'Estado_form': str,
    'Cód. ATC_form': str, 
    'Principios Activos_form': str
})

In [185]:
# Rearrange columns
df_full = df_full.reindex(columns=[
    'Nº Registro', 
    'Cod. Nacional',
    'Código nacional',
    'Cód. ATC_pres',
    'Cód. ATC_form',
    'Situación de financiación',
    'Presentación', 
    'Nombre del medicamento',
    'Medicamento',
    'Principios Activos_pres',
    'Principio activo o asociación*',
    'Principios Activos_form',
    'Nº P. Activos',
    'Tipo de medicamento',
    'Laboratorio_pres',
    'Laboratorio_form',
    'Estado_pres', 
    'Estado_form',
    'Fecha Aut._pres',
    'Fecha Aut._form',
    'Fecha Estado_pres',
    'Fecha Estado_form',
    '¿Comercializado?', 
    '¿Triangulo Amarillo?',
    'Observaciones', 
    '¿Sustituible?', 
    '¿Afecta conducción?',
    '¿Problemas de suministro?', 
    'Más Información'
])


In [191]:
display(df_full.head(6))

,Nº Registro,Cod. Nacional,Presentación,Laboratorio_pres,Fecha Aut._pres,Estado_pres,Fecha Estado_pres,Cód. ATC_pres,Principios Activos_pres,¿Comercializado?,...,Tipo de medicamento,Más Información,Medicamento,Laboratorio_form,Fecha Aut._form,Estado_form,Fecha Estado_form,Cód. ATC_form,Principios Activos_form,Nº P. Activos
0,00-2881,706019,"YASMIN DIARIO 3mg/0,03mg comprimidos recubiert...",Bayer Ab,30/03/2015,Autorizado,30/03/2015,G03AA12,"DROSPIRENONA, ETINILESTRADIOL",True,...,NaN,NaN,"YASMIN DIARIO 3mg/0,03mg comprimidos recubiert...",Bayer Ab,30/03/2015,Autorizado,30/03/2015,G03AA12,"DROSPIRENONA, ETINILESTRADIOL",2
1,00-2881,706020,"YASMIN DIARIO 3mg/0,03mg comprimidos recubiert...",Bayer Ab,30/03/2015,Autorizado,30/03/2015,G03AA12,"DROSPIRENONA, ETINILESTRADIOL",True,...,NaN,NaN,"YASMIN DIARIO 3mg/0,03mg comprimidos recubiert...",Bayer Ab,30/03/2015,Autorizado,30/03/2015,G03AA12,"DROSPIRENONA, ETINILESTRADIOL",2
2,00000235,729183,"BELARA DIARIO 2 MG/0,03 MG COMPRIMIDOS RECUBIE...",Pharma Gerke Arzneimittelvertriebs Gmbh,07/09/2020,Autorizado,07/09/2020,G03AA15,"CLORMADINONA ACETATO, ETINILESTRADIOL",False,...,NaN,NaN,"BELARA DIARIO 2 MG/0,03 MG COMPRIMIDOS RECUBIE...",Pharma Gerke Arzneimittelvertriebs Gmbh,07/09/2020,Autorizado,08/09/2020,G03AA15,"CLORMADINONA ACETATO, ETINILESTRADIOL",2
3,00000235,729184,"BELARA DIARIO 2 MG/0,03 MG COMPRIMIDOS RECUBIE...",Pharma Gerke Arzneimittelvertriebs Gmbh,07/09/2020,Autorizado,07/09/2020,G03AA15,"CLORMADINONA ACETATO, ETINILESTRADIOL",False,...,NaN,NaN,"BELARA DIARIO 2 MG/0,03 MG COMPRIMIDOS RECUBIE...",Pharma Gerke Arzneimittelvertriebs Gmbh,07/09/2020,Autorizado,08/09/2020,G03AA15,"CLORMADINONA ACETATO, ETINILESTRADIOL",2
4,00053-0064,663875,"MICROGYNON COMPRIMIDOS RECUBIERTOS, 21 comprim...",Bayer Plc,21/09/2009,Autorizado,21/09/2009,G03AA07,"LEVONORGESTREL, ETINILESTRADIOL",False,...,NaN,NaN,MICROGYNON COMPRIMIDOS RECUBIERTOS,Bayer Plc,21/09/2009,Autorizado,21/09/2009,G03AA07,"LEVONORGESTREL, ETINILESTRADIOL",2
5,00129001,848226,"AZOPT 10 mg/ml COLIRIO EN SUSPENSION, 1 frasco...",Novartis Europharm Limited,05/05/2000,Autorizado,05/05/2000,S01EC04,BRINZOLAMIDA,True,...,NaN,NaN,AZOPT 10 mg/ml COLIRIO EN SUSPENSION,Novartis Europharm Limited,05/05/2000,Autorizado,05/05/2000,S01EC04,BRINZOLAMIDA,1


In [192]:
# Selecting columns for further analysis
df_full = df_full[df_full["Estado_pres"] == 'Autorizado'][[
    'Código nacional',
    'Cód. ATC_pres',
    'Situación de financiación',
    'Principios Activos_pres',
    'Principio activo o asociación*',
    'Principios Activos_form',
    'Nº P. Activos',
    'Presentación', 
    'Nombre del medicamento',
    'Medicamento',
    'Laboratorio_pres',
    'Estado_pres'
]]

In [193]:
# Save file for LOAD step
df_full.to_csv("loading/df_full.csv", header = True, index = False)